<a href="https://colab.research.google.com/github/minjikim0330/ml_project/blob/main/RendomForest_%EC%84%B1%EB%8A%A5%ED%8F%89%EA%B0%80.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# ==========================================
# 1. 데이터 불러오기 및 통합 (Data Pipeline)
# ==========================================

# [1] 깃허브 데이터 불러오기 (직접 수집한 데이터)
github_raw_url = "https://raw.githubusercontent.com/minjikim0330/ml_project/refs/heads/main/weather_clothing_dataset_1.csv"
df_github = pd.read_csv(github_raw_url)

# =====================================================================
# [2] 구글 드라이브 연결 및 CSV 데이터 불러오기 (수정 완료!)
# =====================================================================
from google.colab import drive

# 1. 구글 드라이브와 코랩을 연결합니다
drive.mount('/content/drive')

excel_file_path = '/content/drive/MyDrive/머신러닝/instagram_style_fashion_100.csv'

df_excel = pd.read_csv(excel_file_path, encoding='cp949')

print("✅ 드라이브 연결 및 인스타 패션 데이터 로드 완벽 성공!")

# # [3] 크롤링 데이터 구조 맞추기
df_excel['outer'] = '없음'
df_excel['inner'] = '없음'

if 'accessory' in df_github.columns:
    df_github = df_github.rename(columns={'accessory': 'accessories'})
if 'accessory' in df_excel.columns:
    df_excel = df_excel.rename(columns={'accessory': 'accessories'})

columns_order = ['temp', 'feel_temp', 'weather', 'gender', 'outer', 'top', 'bottom', 'inner', 'accessories']

df_github = df_github[columns_order]
df_excel = df_excel[columns_order]

# [4] 두 데이터 하나로 합치기 (위아래로 연결)
df = pd.concat([df_github, df_excel], ignore_index=True)
print(f"데이터 통합 완료! 총 데이터 개수: {len(df)}개")

# ==========================================
# [대안 1 적용] 타겟(Y)을 4단계 두께감 카테고리로 변경
# ==========================================


def assign_outfit_level(temp):
    if temp <= 4:
        return "Level 0: 패딩/헤비아우터 + 목도리 (겨울 한파)"
    elif 5 <= temp <= 16:
        return "Level 1: 코트/자켓 + 니트/셔츠 + 긴바지 (쌀쌀한 환절기)"
    elif 17 <= temp <= 22:
        return "Level 2: 맨투맨/가디건 + 청바지/면바지 (선선한 봄/가을)"
    else:
        return "Level 3: 반팔 + 반바지/얇은바지 (더운 여름)"


df['outfit_level'] = df['temp'].apply(assign_outfit_level)

print("--- 변경된 카테고리별 데이터 분포 ---")
print(df['outfit_level'].value_counts())



le_weather = LabelEncoder()
le_gender = LabelEncoder()
le_level = LabelEncoder() # 새로운 라벨 인코더

df['weather_encoded'] = le_weather.fit_transform(df['weather'].astype(str))
df['gender_encoded'] = le_gender.fit_transform(df['gender'].astype(str))
df['target'] = le_level.fit_transform(df['outfit_level'])

# 입력 특성(X)과 새로운 타겟(y) 분리
X = df[['temp', 'feel_temp', 'weather_encoded', 'gender_encoded']]
y = df['target']


# ==========================================
# 3. 모델 학습 및 평가 (Random Forest)
# ==========================================

# 8:2 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 랜덤포레스트 모델 학습
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf_model.fit(X_train, y_train)

# 모델 평가
y_pred = rf_model.predict(X_test)
print("\n=== 새로운 모델 평가 결과 ===")
print(f"테스트 데이터 정확도 (Accuracy): {accuracy_score(y_test, y_pred):.4f}")


# ==========================================
# 4. 새로운 날씨 데이터로 추천 받기 (테스트)
# ==========================================

def predict_my_outfit(temp, feel_temp, weather, gender):
    try:
        # 입력받은 텍스트를 학습된 인코더로 변환
        w_enc = le_weather.transform([weather])[0]
        g_enc = le_gender.transform([gender])[0]

        # 데이터프레임 형태로 입력값 생성
        input_data = pd.DataFrame([[temp, feel_temp, w_enc, g_enc]],
                                  columns=['temp', 'feel_temp', 'weather_encoded', 'gender_encoded'])

        # 예측 및 역변환 (숫자 -> 옷차림 문자열)
        pred_num = rf_model.predict(input_data)
        pred_str = le_outfit.inverse_transform(pred_num)[0]

        print(f"\n[추천 결과] {gender}성 / 기온 {temp}도 (체감 {feel_temp}도) / 날씨: {weather}")
        print(f"👉 추천 코디: {pred_str}")

    except ValueError as e:
        print(f"\n[에러] 학습 데이터에 없는 날씨나 성별이 입력되었습니다: {e}")

# 테스트 실행
# predict_my_outfit(temp=15, feel_temp=14, weather='맑음', gender='남')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 드라이브 연결 및 인스타 패션 데이터 로드 완벽 성공!
데이터 통합 완료! 총 데이터 개수: 10100개
--- 변경된 카테고리별 데이터 분포 ---
outfit_level
Level 0: 패딩/헤비아우터 + 목도리 (겨울 한파)           3286
Level 3: 반팔 + 반바지/얇은바지 (더운 여름)            2870
Level 1: 코트/자켓 + 니트/셔츠 + 긴바지 (쌀쌀한 환절기)    2625
Level 2: 맨투맨/가디건 + 청바지/면바지 (선선한 봄/가을)     1319
Name: count, dtype: int64

=== 새로운 모델 평가 결과 ===
테스트 데이터 정확도 (Accuracy): 1.0000
